## 1 读取负荷数据和光伏数据

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings("ignore")

#设置随机种子
def set_random_seed(seed_value):
    """设置随机种子"""
    np.random.seed(seed_value)  # NumPy
    random.seed(seed_value)  # Python
    torch.manual_seed(seed_value)  # PyTorch CPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)  # PyTorch GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

#设置随机种子，确保结果可复现⭐⭐⭐
set_random_seed(42)
#设置设备
device='cuda' if torch.cuda.is_available() else 'cpu'

#读取数据
data_file = "Annual_load_PV_15min.csv"  # CSV 文件路径,此处为相对路径
df = pd.read_csv(data_file, parse_dates=["Time"])
df.set_index("Time", inplace=True)

pv_profile = df["PV_kW"].values.astype(np.float32)  # 光伏功率序列
load_profile = df["Load_kW"].values.astype(np.float32)  # 负载功率序列

# 爱沙尼亚峰谷电价(€/kWh)
price_profile = np.where((df.index.hour >= 17) & (df.index.hour <= 20),
                         0.30,  # 高峰
                         0.20).astype(np.float32)  # 谷时


## 2 环境定义

In [ ]:
class EnergyStorageEnv(gym.Env):
    def __init__(self, pv, load, price, e_cap=10.0, p_max=3.0, dt=0.25):
        super().__init__()
        self.pv = pv
        self.load = load
        self.price = price
        self.T = len(pv)
        self.dt = dt
        self.e_cap = e_cap
        self.p_max = p_max
        self.soc_min = 0.1
        self.soc_max = 0.9
        self.observation_space = spaces.Box(
            low=np.array([0.,0.,0.], dtype=np.float32),
            high=np.array([1., np.max(load), np.max(pv)], dtype=np.float32),
            dtype=np.float32
        )
        self.action_space = spaces.Discrete(3)

    def reset(self):
        self.t = 0
        self.soc = 0.5
        self.soc_history = []
        self.grid_history = []
        return np.array([self.soc, self.load[self.t], self.pv[self.t]], dtype=np.float32), {}

    def step(self, action):
        p_map = {0:-self.p_max, 1:0.0, 2:self.p_max}
        p_batt = p_map[action]
        self.soc = np.clip(self.soc + p_batt*self.dt/self.e_cap,
                           self.soc_min, self.soc_max)
        net_load = self.load[self.t] - self.pv[self.t] - p_batt
        grid_import = max(net_load, 0)
        reward = -grid_import*self.price[self.t]*self.dt
        self.soc_history.append(self.soc)
        self.grid_history.append(grid_import)
        self.t += 1
        terminated = self.t >= self.T
        truncated = False
        obs = np.array([self.soc,
                        self.load[self.t] if not terminated else 0,
                        self.pv[self.t] if not terminated else 0], dtype=np.float32)
        return obs, reward, terminated, truncated, {}

env = EnergyStorageEnv(pv_profile, load_profile, price_profile)

def plot_training_animation(rewards, soc_history, grid_history,
                                     title="Training Animation",
                                     save_path=None, fps=5):
    """
    Training animation with three separate subplots:
    1. Reward
    2. SOC
    3. Grid Power
    """
    fig, axes = plt.subplots(3,1, figsize=(10,8), sharex=True)

    # Create lines for animation
    line_reward, = axes[0].plot([], [], color='blue', label='Reward', marker='o')
    line_soc, = axes[1].plot([], [], color='green', label='SOC')
    line_grid, = axes[2].plot([], [], color='red', label='Grid Power (kW)')

    # Configure axes
    axes[0].set_ylabel("Reward")
    axes[0].grid(True)
    axes[0].set_title("Training Reward")

    axes[1].set_ylabel("SOC")
    axes[1].grid(True)
    axes[1].set_title("Battery SOC")

    axes[2].set_ylabel("Grid Power (kW)")
    axes[2].set_xlabel("Episode / Time Step")
    axes[2].grid(True)
    axes[2].set_title("Grid Power")

    fig.suptitle(title, fontsize=16, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # Initialization function
    def init():
        line_reward.set_data([], [])
        line_soc.set_data([], [])
        line_grid.set_data([], [])
        return line_reward, line_soc, line_grid

    # Update function for animation
    def update(i):
        x = range(i+1)
        line_reward.set_data(x, rewards[:i+1])
        line_soc.set_data(x, soc_history[:i+1])
        line_grid.set_data(x, grid_history[:i+1])
        return line_reward, line_soc, line_grid

    # Create animation
    ani = FuncAnimation(fig, update, frames=len(rewards),
                        init_func=init, interval=200, blit=True)

    # Display in Jupyter
    display(HTML(ani.to_jshtml()))
    plt.close(fig) 

    # Optional save
    if save_path:
        ani.save(save_path, writer="ffmpeg", fps=fps)


## 3 Q-Learning 算法

In [ ]:
def plot_reward_live(rewards, title="Live Reward"):
    plt.figure(figsize=(8,4))
    plt.plot(rewards, marker='o', color='blue')
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(True)
    plt.show()


soc_bins = np.linspace(env.soc_min, env.soc_max, 8)
load_bins = np.linspace(0, max(load_profile), 12)
pv_bins = np.linspace(0, max(pv_profile), 12)

def discretize_state(s):
    return (np.digitize(s[0], soc_bins)-1,
            np.digitize(s[1], load_bins)-1,
            np.digitize(s[2], pv_bins)-1)

q_table = np.zeros((8,12,12,3))
lr, gamma, eps = 0.1, 0.99, 0.1
q_rewards, q_soc_history, q_grid_history = [], [], []

total_episodes = 50
for ep in range(total_episodes):
    s, _ = env.reset()
    idx = discretize_state(s)
    done = False
    total_reward = 0
    env.soc_history, env.grid_history = [], []
    while not done:
        a = env.action_space.sample() if random.random()<eps else np.argmax(q_table[idx])
        ns, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        nidx = discretize_state(ns)
        q_table[idx][a] += lr*(r + gamma*np.max(q_table[nidx]) - q_table[idx][a])
        idx = nidx
        total_reward += r
    q_rewards.append(total_reward)
    q_soc_history = env.soc_history.copy()
    q_grid_history = env.grid_history.copy()

    # 每段 episode 更新奖励动态
    clear_output(wait=True)
    plot_reward_live(q_rewards, title="Q-Learning Live Reward")

## 4 DQN算法

In [ ]:
class DQNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,3))
    def forward(self,x):
        return self.fc(x.float())

dqn_net = DQNNet()
target_net = DQNNet()
target_net.load_state_dict(dqn_net.state_dict())
optimizer = optim.Adam(dqn_net.parameters(), lr=1e-3)
memory = deque(maxlen=5000)
batch_size = 32
dqn_rewards, dqn_soc_history, dqn_grid_history = [], [], []

for ep in range(total_episodes):
    s, _ = env.reset()
    done = False
    total_reward = 0
    env.soc_history, env.grid_history = [], []
    while not done:
        a = env.action_space.sample() if random.random()<eps else torch.argmax(dqn_net(torch.tensor(s, dtype=torch.float32))).item()
        ns, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        ns_tensor = torch.tensor(ns, dtype=torch.float32)
        memory.append((torch.tensor(s, dtype=torch.float32), a, r, ns_tensor, done))
        s = ns
        total_reward += r
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards_batch, next_states, dones = zip(*batch)
            states = torch.stack(states).float()
            next_states = torch.stack(next_states).float()
            actions = torch.tensor(actions)
            rewards_batch = torch.tensor(rewards_batch)
            dones = torch.tensor(dones, dtype=torch.float32)
            q_values = dqn_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                q_next = target_net(next_states).max(1)[0]
            target = rewards_batch + gamma*q_next*(1-dones)
            loss = nn.MSELoss()(q_values, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    dqn_rewards.append(total_reward)
    dqn_soc_history = env.soc_history.copy()
    dqn_grid_history = env.grid_history.copy()
    clear_output(wait=True)
    plot_reward_live(dqn_rewards, title="DQN Live Reward")

## 5 Policy Gradient

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,3), nn.Softmax(dim=-1))
    def forward(self,x):
        return self.fc(x.float())

policy_net = PolicyNet()
optimizer_pg = optim.Adam(policy_net.parameters(), lr=1e-3)
pg_rewards, pg_soc_history, pg_grid_history = [], [], []

for ep in range(total_episodes):
    s, _ = env.reset()
    done = False
    log_probs, rewards_list = [], []
    total_reward = 0
    env.soc_history, env.grid_history = [], []
    while not done:
        probs = policy_net(torch.tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        ns, r, terminated, truncated, _ = env.step(a.item())
        done = terminated or truncated
        log_probs.append(dist.log_prob(a))
        rewards_list.append(r)
        s = ns
        total_reward += r
    # 计算回报
    G = 0
    returns = []
    for r in reversed(rewards_list):
        G = r + gamma*G
        returns.insert(0,G)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean())/(returns.std()+1e-9)
    loss = 0
    for lp, R in zip(log_probs, returns):
        loss -= lp*R
    optimizer_pg.zero_grad()
    loss.backward()
    optimizer_pg.step()
    pg_rewards.append(total_reward)
    pg_soc_history = env.soc_history.copy()
    pg_grid_history = env.grid_history.copy()
    clear_output(wait=True)
    plot_reward_live(pg_rewards, title="Policy Gradient Live Reward")

## 6 Actor-Critic(AC) 演员-评论家算法

In [ ]:
class ACNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(3,64)
        self.actor = nn.Linear(64,3)
        self.critic = nn.Linear(64,1)
    def forward(self,x):
        x = torch.relu(self.fc(x.float()))
        return torch.softmax(self.actor(x),-1), self.critic(x)

ac_net = ACNet()
optimizer_ac = optim.Adam(ac_net.parameters(), lr=1e-3)
ac_rewards, ac_soc_history, ac_grid_history = [], [], []

for ep in range(total_episodes):
    s, _ = env.reset()
    done = False
    total_reward = 0
    env.soc_history, env.grid_history = [], []
    while not done:
        probs, value = ac_net(torch.tensor(s, dtype=torch.float32))
        dist = torch.distributions.Categorical(probs)
        a = dist.sample()
        ns, r, terminated, truncated, _ = env.step(a.item())
        done = terminated or truncated
        _, next_value = ac_net(torch.tensor(ns, dtype=torch.float32))
        td_target = r + gamma*next_value*(1-done)
        td_error = td_target - value
        loss = -dist.log_prob(a)*td_error.detach() + td_error.pow(2)
        optimizer_ac.zero_grad()
        loss.backward()
        optimizer_ac.step()
        s = ns
        total_reward += r
    ac_rewards.append(total_reward)
    ac_soc_history = env.soc_history.copy()
    ac_grid_history = env.grid_history.copy()
    clear_output(wait=True)
    plot_reward_live(ac_rewards, title="Actor-Critic Live Reward")

## 7 可视化

In [ ]:
plot_training_animation(q_rewards, q_soc_history, q_grid_history, title="Q-Learning Animation")
plot_training_animation(dqn_rewards, dqn_soc_history, dqn_grid_history, title="DQN Animation")
plot_training_animation(pg_rewards, pg_soc_history, pg_grid_history, title="Policy Gradient Animation")
plot_training_animation(ac_rewards, ac_soc_history, ac_grid_history, title="Actor-Critic Animation")
